In [1]:
%autosave 60
%pip install --quiet -r requirements.txt

Autosaving every 60 seconds
Note: you may need to restart the kernel to use updated packages.


## Setup

In [2]:
import os
import numpy as np
import random
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import defaultdict
from utils.hunt_data_loader import HuntDataLoader
from utils.train_eval import fit_3D, fit_3D_gan
from utils.loss_functions import recon_loss, ssim_L1_2d_loss, ssim_loss_3d, l1_loss
from models.alzheiminator_3d import ResidualUNet3D, Discriminator3D, Generator3D
from models.no_more_alzheimer_2d import UNet2D
from tqdm import tqdm

data_loader = HuntDataLoader()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Load models

We load our best performing 2D model

In [3]:
best_2d = UNet2D(latent_dim=64, in_channels=1, out_channels=1, base_ch=32).to(device)
best_2d.load_state_dict(torch.load("out/unet_models/2d_unet_model_best.pt"))

<All keys matched successfully>

We load our best performing 3D model

In [4]:
best_3d = ResidualUNet3D(in_ch=1, base=32).to(device)
best_3d.load_state_dict(torch.load("out/unet_models/3d_unet_model_best.pt"))

<All keys matched successfully>

## Display evenly spaced slices for both models

In [5]:
# TODO

## Calculate Loss for each model

In [6]:
# Same split as during training
_, _, test_pairs = data_loader.split_dataset_paths(seed=69)
print("We have", len(test_pairs), "test pairs")

We have 71 test pairs


Helper function taking a 3D input sensor and running a 2D model over all slices, and then combining them into an output volume

In [7]:
def get_volume_from_2d_pred(model, x_vol):
    model.eval()

    # --- Normalize input shape to (D, H, W) ---
    # (1, 1, D, H, W) -> (D, H, W)
    x_vol = x_vol.squeeze(0).squeeze(0)
    D = x_vol.shape[0]

    output_vol = None  # will be allocated from first output slice

    with torch.no_grad():
        for i in range(D):
            # (H, W) -> (1,1,H,W)
            x_slice = x_vol[i].unsqueeze(0).unsqueeze(0).to(device)

            # Forward pass; adapt unpacking if your model returns differently
            recon, _, _ = model(x_slice)    # recon: (1,1,H_out,W_out)
            recon_slice = recon[0, 0]       # (H_out, W_out)

            if output_vol is None:
                H_out, W_out = recon_slice.shape
                output_vol = torch.zeros(
                    (D, H_out, W_out),
                    dtype=recon_slice.dtype,
                    device=device,
                )

            output_vol[i] = recon_slice

    # Return (1,1,D,H_out,W_out)
    return output_vol.unsqueeze(0).unsqueeze(0)

In [8]:
def center_crop_to_smallest(*vols):
    """
    Center-crop all volumes (B,C,D,H,W) to the smallest common (D,H,W).
    Assumes all volumes have same B and C.
    """
    # Get min spatial sizes
    Ds, Hs, Ws = zip(*(v.shape[2:] for v in vols))
    D_min, H_min, W_min = min(Ds), min(Hs), min(Ws)

    cropped = []
    for v in vols:
        _, _, D, H, W = v.shape
        d_start = (D - D_min) // 2
        h_start = (H - H_min) // 2
        w_start = (W - W_min) // 2
        cropped.append(
            v[:, :, d_start:d_start + D_min,
                 h_start:h_start + H_min,
                 w_start:w_start + W_min]
        )
    return cropped

Calculate average loss over the entire test set

In [9]:
best_2d.eval()
best_3d.eval()

num_pairs = len(test_pairs)

error_metrics = [
    ("ssim_3D", ssim_loss_3d),
    ("L1", l1_loss),
    ("ssim_l1", recon_loss),
]

average_2d_losses = np.zeros(len(error_metrics))
average_3d_losses = np.zeros(len(error_metrics))

with torch.no_grad():
    for idx, (hunt3_path, hunt4_path) in enumerate(tqdm(test_pairs, total=num_pairs)):
        # Load volumes, maybe as (H, W, D) or (D, H, W)
        x_vol_np = data_loader.load_from_path(hunt3_path, crop_size=(192, 224))
        y_vol_np = data_loader.load_from_path(hunt4_path, crop_size=(192, 224))

        # Ensure (D, H, W)
        if x_vol_np.shape[0] != 192 and x_vol_np.shape[2] == 192:
            x_vol_np = np.transpose(x_vol_np, (2, 0, 1))
            y_vol_np = np.transpose(y_vol_np, (2, 0, 1))

        # To torch: (D, H, W)
        x_vol = torch.from_numpy(x_vol_np).float().to(device)
        y_vol = torch.from_numpy(y_vol_np).float().to(device)

        D, H, W = x_vol.shape

        # ---------- 2D model: slice-by-slice ----------
        recon_2d_slices = []
        for d in range(D):
            # (H, W) -> (1, 1, H, W)
            x_slice = x_vol[d].unsqueeze(0).unsqueeze(0)
            recon, _, _ = best_2d(x_slice)  # recon: (1, 1, H_out, W_out)
            recon_2d_slices.append(recon[0, 0])  # (H_out, W_out)

        recon_2d_vol = torch.stack(recon_2d_slices, dim=0)  # (D, H2, W2)

        # ---------- 3D model: full volume ----------
        x_vol_5d = x_vol.unsqueeze(0).unsqueeze(0)  # (1, 1, D, H, W)
        recon_3d_vol, _ = best_3d(x_vol_5d)         # (1, 1, D3, H3, W3)
        recon_3d_vol = recon_3d_vol[0, 0]           # (D3, H3, W3)

        # ---------- Center-crop all to smallest (D,H,W) ----------
        D2, H2, W2 = recon_2d_vol.shape
        D3, H3, W3 = recon_3d_vol.shape
        Dy, Hy, Wy = y_vol.shape

        D_min = min(D2, D3, Dy)
        H_min = min(H2, H3, Hy)
        W_min = min(W2, W3, Wy)

        # 2D recon
        d0 = (D2 - D_min) // 2
        h0 = (H2 - H_min) // 2
        w0 = (W2 - W_min) // 2
        recon_2d_vol = recon_2d_vol[
            d0:d0 + D_min,
            h0:h0 + H_min,
            w0:w0 + W_min
        ]

        # 3D recon
        d0 = (D3 - D_min) // 2
        h0 = (H3 - H_min) // 2
        w0 = (W3 - W_min) // 2
        recon_3d_vol = recon_3d_vol[
            d0:d0 + D_min,
            h0:h0 + H_min,
            w0:w0 + W_min
        ]

        # Ground truth
        d0 = (Dy - D_min) // 2
        h0 = (Hy - H_min) // 2
        w0 = (Wy - W_min) // 2
        y_vol_aligned = y_vol[
            d0:d0 + D_min,
            h0:h0 + H_min,
            w0:w0 + W_min
        ]

        # Add (B, C) dims back: (1,1,D,H,W)
        recon_2d_vol = recon_2d_vol.unsqueeze(0).unsqueeze(0)
        recon_3d_vol = recon_3d_vol.unsqueeze(0).unsqueeze(0)
        y_vol_aligned = y_vol_aligned.unsqueeze(0).unsqueeze(0)

        # ---------- Compute losses ----------
        for k, (_, metric_fn) in enumerate(error_metrics):
            average_2d_losses[k] += metric_fn(recon_2d_vol, y_vol_aligned).item()
            average_3d_losses[k] += metric_fn(recon_3d_vol, y_vol_aligned).item()

# Average
average_2d_losses /= num_pairs
average_3d_losses /= num_pairs

# Print
for k, (metric_name, _) in enumerate(error_metrics):
    print(f" ------ Metric: {metric_name} ------ ")
    print(f"  Average 2D U-Net Error:   {average_2d_losses[k]:.4f}")
    print(f"  Average 3D U-Net Error:   {average_3d_losses[k]:.4f}")
    print()

  0%|          | 0/71 [00:00<?, ?it/s]

100%|██████████| 71/71 [01:05<00:00,  1.09it/s]

 ------ Metric: ssim_3D ------ 
  Average 2D U-Net Error:   0.1197
  Average 3D U-Net Error:   0.1378

 ------ Metric: L1 ------ 
  Average 2D U-Net Error:   0.0133
  Average 3D U-Net Error:   0.0167

 ------ Metric: ssim_l1 ------ 
  Average 2D U-Net Error:   0.0665
  Average 3D U-Net Error:   0.0773



## Compare loss over entire volume for both models

As the 2D model only generates slices, it has to be run individually over all slices in a volume